# Spatial Panel Time-Series Analysis of Urban Morphology  
## Subdivision–NPA Aggregation and Fixed-Effects Modeling (1990–2023)

This notebook constructs a **spatio-temporal panel dataset** by aggregating
subdivision-level urban morphology indicators to **dominant Neighborhood Profile Areas (NPAs)**.

The workflow includes:
- Spatial intersection and area-weighted attribution of subdivisions to NPAs
- Construction of dominance and fragmentation indicators
- Panel aggregation and diagnostics
- Fixed-effects, two-way fixed-effects, and dynamic panel models
- Structural break detection in long-run trends

The final objective is to assess **temporal trends and regime shifts** in urban morphology
while controlling for unobserved spatial heterogeneity.


## 1. Environment Setup and Imports

We begin by importing core data science, geospatial, and visualization libraries.
All numeric outputs are formatted for readability.


In [17]:
import pandas as pd
import geopandas as gpd
import folium
from arch.unitroot import ADF, KPSS
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import ruptures as rpt
import numpy as np
from IPython.display import HTML

pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
ABT = gpd.read_file(
    "../../../../Data/Final_dataset/ABT/ABT.gpkg",
    layer="subdivisions"
)

npa_raw = gpd.read_file(
    "../../../../Data/Original_dataset/original.gdb",
    layer="QOL_NPA_2020_final_projected"
)

## 3. Temporal Filtering and Coordinate Alignment

We restrict the analysis to the study period (1990–2023) and ensure both layers
share a common projected coordinate reference system for area calculations.


In [3]:
ABT_proj = ABT[ABT["year"].between(1990, 2023)].copy()
npa_proj = npa_raw.to_crs(ABT_proj.crs)

ABT_proj["subd_area"] = ABT_proj.geometry.area

## 4. Spatial Overlay: Subdivision × NPA

Each subdivision may intersect multiple NPAs.
We compute intersection geometries and their respective areas.


In [4]:
abt_npa_intersections = gpd.overlay(
    ABT_proj[["subd_id", "geometry"]],
    npa_proj[["NPA_ID", "geometry"]],
    how="intersection"
)

abt_npa_intersections["intersect_area"] = (
    abt_npa_intersections.geometry.area
)

## 5. Area Shares, NPA Counts, and Fragmentation Structure

For each subdivision:
- Count how many NPAs it intersects
- Compute area-weighted shares
- Rank NPAs by dominance


In [5]:
npa_area = (
    abt_npa_intersections
    .groupby(["subd_id", "NPA_ID"])["intersect_area"]
    .sum()
    .reset_index()
)

npa_count = (
    npa_area
    .groupby("subd_id")["NPA_ID"]
    .nunique()
    .reset_index(name="npa_count")
)

npa_id_list = (
    npa_area
    .groupby("subd_id")["NPA_ID"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .reset_index(name="npa_id_list")
)


## 6. Dominant NPA Ranking and Share Validation

We compute the proportional area share of each NPA per subdivision
and verify that all shares sum to unity.


In [6]:
npa_area = npa_area.merge(
    ABT_proj[["subd_id", "subd_area"]],
    on="subd_id",
    how="left"
)

npa_area["share"] = (
    npa_area["intersect_area"] / npa_area["subd_area"]
)

npa_area = npa_area.sort_values(
    ["subd_id", "share"],
    ascending=[True, False]
)

npa_area["rank"] = (
    npa_area
    .groupby("subd_id")
    .cumcount() + 1
)


## 7. Wide-Format Share Matrix (Top 6 NPAs)

To maintain interpretability and avoid extreme fragmentation,
we retain the top 6 NPAs per subdivision and aggregate the remainder.


In [7]:
npa_area_top = npa_area[npa_area["rank"] <= 6].copy()

npa_share_wide = (
    npa_area_top
    .pivot_table(
        index="subd_id",
        columns="rank",
        values="share",
        fill_value=0
    )
)

npa_share_wide.columns = [
    f"share_npa_{int(c)}" for c in npa_share_wide.columns
]

npa_share_wide = npa_share_wide.reset_index()


## 8. Final Share Assembly and Consistency Check

We compute the residual share outside NPAs and verify numerical integrity.


In [8]:
ABT_proj = (
    ABT_proj
    .merge(npa_share_wide, on="subd_id", how="left")
    .merge(npa_count, on="subd_id", how="left")
    .merge(npa_id_list, on="subd_id", how="left")
)

share_cols = [f"share_npa_{k}" for k in range(1, 7)]

ABT_proj[share_cols] = ABT_proj[share_cols].fillna(0)
ABT_proj["npa_count"] = ABT_proj["npa_count"].fillna(0).astype(int)

ABT_proj["share_outside_npa"] = (
    1 - ABT_proj[share_cols].sum(axis=1)
).clip(lower=0)

ABT_proj["share_sum_check"] = (
    ABT_proj[share_cols + ["share_outside_npa"]].sum(axis=1)
)

assert ABT_proj["share_sum_check"].between(0.999, 1.001).all()


## 9. Dominant NPA Assignment

Each subdivision is assigned to its **dominant NPA**
based on maximum area overlap.


In [9]:
dominant_npa = (
    npa_area
    .sort_values(["subd_id", "share"], ascending=[True, False])
    .groupby("subd_id", as_index=False)
    .first()[["subd_id", "NPA_ID", "share"]]
    .rename(columns={
        "NPA_ID": "dominant_npa_id",
        "share": "dominant_npa_share"
    })
)

ABT_proj = ABT_proj.merge(
    dominant_npa,
    on="subd_id",
    how="left"
)


## 10. Panel Construction (NPA × Year)

We aggregate subdivision-level indicators to the NPA–year level
using simple means.


In [20]:
ABT_proj_npa_ready = ABT_proj[["year", "dominant_npa_id",'HAC_dist', 'BAD', 'SHD', 'int_den025','nd_deg025', 'int_den05', 'nd_deg05', 'int_den075', 'nd_deg075',
       'int_den1', 'nd_deg1', 'AI', 'PROX', 'ENN_MN', 'ED', 'SHAPE_MN',
       'FRAC_MN', 'ENN_inv', 'ED_inv', 'SHAPE_inv', 'FRAC_inv', 'AI_norm',
       'PROX_norm', 'ENN_inv_norm', 'ED_inv_norm', 'SHAPE_inv_norm',
       'FRAC_inv_norm', 'COMPACTNESS_SUM', 'BAD_ctx_025', 'BAD_ctx_050',
       'groceries_ws', 'transit_ws', 'FAR']]

panel = (
    ABT_proj_npa_ready
    .groupby(["dominant_npa_id", "year"], as_index=True)
    .mean(numeric_only=True)
    .sort_index()
)


In [21]:
panel

HAC_dist    BAD    SHD  int_den025  nd_deg025  \
dominant_npa_id year                                                       
2.0000          1996.0000    1.8000 0.1940 0.4000      0.1164     2.0465   
                1999.0000    1.5550 0.2320 0.0000      0.1053     2.0263   
                2008.0000    1.8600 0.2120 0.2200      0.1011     2.0690   
                2009.0000    2.0200 0.2290 0.0000      0.0818     2.0741   
                2013.0000    1.7000 0.1110 0.0000      0.1208     2.0541   
...                             ...    ...    ...         ...        ...   
476.0000        2018.0000    0.5367 0.5322 0.7250      0.1617     2.4448   
                2019.0000    0.5200 0.5639 0.2943      0.1567     2.4690   
                2020.0000    0.3200 0.7290 1.0000      0.2111     2.6333   
                2022.0000    0.5633 0.4897 0.4000      0.2346     2.6274   
                2023.0000    0.6400 0.3305 0.3500      0.1084     2.2863   

                           int_den05  nd_deg05  int_den075  nd_deg075  \
dominant_npa_id year                                                    
2.0000          1996.0000     0.0845    2.1702      0.1013     2.3568   
                1999.0000     0.0817    2.1794      0.0997     2.3157   
                2008.0000     0.0823    2.1628      0.1036     2.3689   
                2009.0000     0.0882    2.3133      0.1158     2.3556   
                2013.0000     0.0693    2.1053      0.1001     2.2885   
...                              ...       ...         ...        ...   
476.0000        2018.0000     0.1453    2.6527      0.1306     2.7439   
                2019.0000     0.1413    2.6444      0.1398     2.7600   
                2020.0000     0.1799    2.8333      0.1414     2.8046   
                2022.0000     0.1871    2.6880      0.1702     2.7926   
                2023.0000     0.1081    2.5425      0.1326     2.6414   

                           int_den1  ...  ENN_inv_norm  ED_inv_norm  \
dominant_npa_id year                 ...                              
2.0000          1996.0000    0.1020  ...        0.1054       0.0026   
                1999.0000    0.1205  ...        0.0890       0.0029   
                2008.0000    0.1049  ...        0.0483       0.0022   
                2009.0000    0.1043  ...        0.1304       0.0019   
                2013.0000    0.1062  ...        0.0289       0.0043   
...                             ...  ...           ...          ...   
476.0000        2018.0000    0.1266  ...        0.6861       0.0072   
                2019.0000    0.1386  ...        0.3294       0.0028   
                2020.0000    0.1358  ...        1.0000       0.0031   
                2022.0000    0.1660  ...        0.6840       0.0037   
                2023.0000    0.1266  ...        0.0388       0.0040   

                           SHAPE_inv_norm  FRAC_inv_norm  COMPACTNESS_SUM  \
dominant_npa_id year                                                        
2.0000          1996.0000          0.4997         0.6063           0.0479   
                1999.0000          0.4094         0.5451           0.0461   
                2008.0000          0.8772         0.8751           0.0301   
                2009.0000          0.8757         0.8644           0.0582   
                2013.0000          0.8587         0.8498           0.0232   
...                                   ...            ...              ...   
476.0000        2018.0000          0.6988         0.7822           0.2875   
                2019.0000          0.8246         0.8341           0.1492   
                2020.0000          0.5342         0.7091           0.4309   
                2022.0000          0.7455         0.8088           0.2753   
                2023.0000          0.9151         0.9053           0.0342   

                           BAD_ctx_025  BAD_ctx_050  groceries_ws  transit_ws  \
dominant_npa_id year                                                            


## 11. Stationarity Diagnostics (Panel Mean Series)

We test whether long-run average morphology indicators
exhibit unit roots.


In [22]:
mean_series = panel.groupby("year")["HAC_dist"].mean()

ADF(mean_series).summary()
KPSS(mean_series).summary()

C:\Users\erfan\AppData\Local\Temp\ipykernel_2924\4180925931.py:4: DeprecationWarning: Lag selection has changed to use a data-dependent method. To use the old method that only depends on time, set lags=-1
  KPSS(mean_series).summary()


Test Statistic,0.173
P-value,0.326
Lags,2


## 12. Fixed-Effects Time Trend Models

We estimate within-NPA temporal trends using entity fixed effects
and clustered standard errors.


In [23]:
def fe_trend(panel, yvar):
    y = panel[yvar]
    X = pd.DataFrame(
        {"year": panel.index.get_level_values("year")},
        index=panel.index
    )
    X = sm.add_constant(X)

    return PanelOLS(
        y, X, entity_effects=True
    ).fit(cov_type="clustered", cluster_entity=True)

fe_trend(panel, "HAC_dist").summary

Dep. Variable:,HAC_dist,R-squared:,0.0049
Estimator:,PanelOLS,R-squared (Between):,-0.0002
No. Observations:,3998,R-squared (Within):,0.0049
Date:,"Thu, Feb 05 2026",R-squared (Overall):,-0.0002
Time:,23:05:07,Log-likelihood,-1694.6
Cov. Estimator:,Clustered,,
,,F-statistic:,17.517
Entities:,443,P-value,0.0000
Avg Obs:,9.0248,Distribution:,"F(1,3554)"
Min Obs:,1.0000,,
Max Obs:,29.000,F-statistic (robust):,13.340


## 13. Two-Way Fixed Effects (Entity + Time)

This specification absorbs **common temporal shocks**
affecting all NPAs simultaneously.


In [16]:
def fe_tw(panel, yvar):
    y = panel[yvar]
    X = pd.DataFrame(
        {"year": panel.index.get_level_values("year")},
        index=panel.index
    )
    X = sm.add_constant(X)

    return PanelOLS(
        y, X,
        entity_effects=True,
        time_effects=True
    ).fit(cov_type="clustered", cluster_entity=True)

fe_tw(panel, "HAC_dist").summary

AbsorbingEffectError: 
The model cannot be estimated. The included effects have fully absorbed
one or more of the variables. This occurs when one or more of the dependent
variable is perfectly explained using the effects included in the model.

The following variables or variable combinations have been fully absorbed
or have become perfectly collinear after effects are removed:

          const, year

Set drop_absorbed=True to automatically drop absorbed variables.


## 14. Structural Break Detection in Mean Trends

We apply PELT with an RBF cost function to identify
regime shifts in long-run morphology trends.


In [18]:
mean_ts = panel.reset_index().groupby("year").mean(numeric_only=True)
y_std = (mean_ts["HAC_dist"] - mean_ts["HAC_dist"].mean()) / mean_ts["HAC_dist"].std()

algo = rpt.Pelt(model="rbf").fit(y_std.values)
breaks = algo.predict(pen=3 * np.log(len(y_std)))

mean_ts.index.values[np.array(breaks)[:-1]]

array([], dtype=float64)

## 15. Dynamic Panel Model (Lagged Dependence)

We test persistence in urban morphology using
a lagged dependent variable specification.


In [19]:
panel_dyn = panel.copy()
panel_dyn["HAC_L1"] = (
    panel_dyn.groupby(level=0)["HAC_dist"].shift(1)
)

panel_dyn = panel_dyn.dropna()

y = panel_dyn["HAC_dist"]
X = panel_dyn[["HAC_L1"]]
X["year"] = panel_dyn.index.get_level_values("year")
X = sm.add_constant(X)

PanelOLS(
    y, X, entity_effects=True
).fit(cov_type="clustered", cluster_entity=True).summary

C:\Users\erfan\AppData\Local\Temp\ipykernel_2924\841297523.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["year"] = panel_dyn.index.get_level_values("year")


Dep. Variable:,HAC_dist,R-squared:,0.0049
Estimator:,PanelOLS,R-squared (Between):,-0.0728
No. Observations:,3555,R-squared (Within):,0.0049
Date:,"Thu, Feb 05 2026",R-squared (Overall):,-0.0719
Time:,22:12:44,Log-likelihood,-1533.7
Cov. Estimator:,Clustered,,
,,F-statistic:,7.6460
Entities:,419,P-value,0.0005
Avg Obs:,8.4845,Distribution:,"F(2,3134)"
Min Obs:,1.0000,,
Max Obs:,28.000,F-statistic (robust):,5.1520


In [ ]:
!jupyter nbconvert --to html --no-input EDA.ipynb --output ../../../../output/Notebook_Outputs/spatio_temporal/EDA.html